PDF 문서 기반 QA(Question-Answer)

1. 1단계 문서로드(Document Load): 문서 내용을 불러옵니다.
2. 2단계 분할(Text Split): 문서를 특정 기준(Chunk) 으로 분할합니다.
3. 3단계 임베딩(Embedding): 분할된(Chunk) 를 임베딩하여 저장합니다.
4. 4단계 벡터DB 저장: 임베딩된 Chunk 를 DB에 저장합니다.
5. 5단계 검색기(Retriever): 쿼리(Query) 를 바탕으로 DB에서 검색하여 결과를 가져오기 위하여 리트리버를 정의합니다. 리트리버는 검색 알고리즘이며(Dense, Sparse) 리트리버로 나뉘게 됩니다. Dense: 유사도 기반 검색, Sparse: 키워드 기반 검색
6. 6단계 프롬프트: RAG 를 수행하기 위한 프롬프트를 생성합니다. 프롬프트의 context 에는 문서에서 검색된 내용이 입력됩니다. 프롬프트 엔지니어링을 통하여 답변의 형식을 지정할 수 있습니다.
7. 7단계 LLM: 모델을 정의합니다.(GPT-3.5, GPT-4, Claude, etc..)
8. 8단계 Chain: 프롬프트 - LLM - 출력 에 이르는 체인을 생성합니다.

In [1]:
# =========================================================
# 0) 라이브러리 설치 (필수)
# =========================================================
# 역할:
# LangChain, PDF loader, OpenAI, FAISS 등 RAG 구성 필수 패키지 설치

!pip install -q langchain langchain-openai langchain-community langchain-text-splitters faiss-cpu pypdf



In [20]:

from google.colab import userdata
import os

os.environ['OPENAI_API_KEY'] = userdata.get("OPENAI_API_KEY")

# =========================================================
# 0) 로컬 PDF 파일 업로드 (추가된 핵심 부분)
# =========================================================
# 역할:
# 로컬 PC에 있는 PDF 파일을 코드 실행 환경으로 업로드

from google.colab import files  # Colab 전용 (로컬 업로드 기능)

uploaded = files.upload()
# 실행 시 파일 선택 창이 뜸
# 업로드된 파일은 현재 작업 디렉토리에 저장됨

# 업로드된 파일명 자동 추출
pdf_file_name = list(uploaded.keys())[0]


# =========================================================
# 1) Document Load (PDF 로드 단계)
# =========================================================
# 역할:
# 업로드된 PDF 파일을 Document 객체로 변환

from langchain_community.document_loaders import PyPDFLoader

loader =

# 로컬에서 업로드된 PDF 파일을 그대로 사용

docs =

# 페이지 단위 Document 리스트 생성


# =========================================================
# 2) Text Split (Chunk 분할 단계)
# =========================================================
# 역할:
# 긴 문서를 검색 가능한 작은 단위로 분할

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(

    # 문서 의미를 유지하면서도 검색 가능한 크기

    # 문맥 끊김 방지
)

splits =
# Document → chunk 리스트로 변환


# =========================================================
# 3) Embedding 생성
# =========================================================
# 역할:
# 텍스트를 벡터로 변환하여 의미 검색 가능하게 만듦

from langchain_openai import OpenAIEmbeddings

embeddings =
# OpenAI embedding 모델 초기화


# =========================================================
# 4) Vector DB 생성 (FAISS)
# =========================================================
# 역할:
# chunk embedding을 저장하고 빠르게 검색

from langchain_community.vectorstores import FAISS

vectorstore =



vectorstore.save_local("faiss_index")
# 선택: 로컬 저장 (재사용 가능)


# =========================================================
# 5) Retriever 정의
# =========================================================
# 역할:
# 질문을 기반으로 관련 문서 검색

retriever =


# top-k 유사 문서 반환


# =========================================================
# 6) Prompt 설계
# =========================================================
# 역할:
# 검색된 context를 LLM에게 전달하는 규칙 정의

from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
너는 PDF 문서를 분석하는 AI이다.

아래 Context를 근거로 질문에 답변하라.

답변에는 Context의 내용을 요약하거나 인용하여 설명하라.

Context에 관련 내용이 전혀 없는 경우에만
"문서에 없음"이라고 답변한다.

-------------------------
Context:
{context}

Question:
{question}

-------------------------
Answer:
""")


# =========================================================
# 7) LLM 정의
# =========================================================
# 역할:
# 최종 답변 생성 모델

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)


# =========================================================
# 8) LCEL Chain 구성
# =========================================================
# 역할:
# retriever → prompt → llm → output 연결

from langchain_core.output_parsers import StrOutputParser

# Document → string 변환 함수
def format_docs(docs):

    return


rag_chain =




# =========================================================
# 9) 실행 테스트
# =========================================================

result = rag_chain.invoke("이 문서의 내용을 설명해줘")
print(result)

Saving ★ 서울특별시 스마트도시 및 정보화 기본계획(홈페이지 게시용).pdf to ★ 서울특별시 스마트도시 및 정보화 기본계획(홈페이지 게시용) (5).pdf
이 문서는 서울시의 스마트도시 조성 및 정보화 기본계획에 관한 내용으로, 시민의 디지털 안전망 강화와 스마트 인프라 구축을 중심으로 하고 있습니다. 주요 내용은 다음과 같습니다.

- 공공와이파이 2만여 대 구축 및 운영을 통해 시민들이 LTE보다 빠른 WiFi6 기반 통신환경을 이용할 수 있도록 하며, 통합관리센터 구축과 보안 접속, SSID 일원화로 서울 전역에서 자동 연결이 가능하도록 하여 시민 이용 편의를 높이고 있습니다.
- 3D 가시권 시뮬레이션, 바람길 구현, 화재 예방 등 안전 관련 스마트 기술도 포함되어 있습니다.
- 문서 목차에는 추진 근거 및 배경, 국내외 동향, 서울시 주요 성과 및 정책 동향, 현황 분석 및 추진 방향, 비전 및 추진 전략, 세부 과제, 투자 계획 및 성과 지표, 향후 계획 등이 체계적으로 정리되어 있습니다.
- 미래 스마트도시 혁신 기반 조성, 사람 중심 스마트도시 구현, 시민 체감 도시 서비스 제공 등 세 가지 큰 축으로 전략이 구성되어 있으며, 비대면 서비스 확대, 스마트 포용 도시 실현, 사이버 안전 도시 실현, 스마트 모빌리티 기반 구축, 안전·안심 도시 서비스 제공, 디지털 경제 활성화 지원 등이 주요 세부 과제로 제시되어 있습니다.
- 또한, 인간 중심(People-Centric), 초자동화(Hyperautomation), 행동 인터넷(Internet of behaviors), 다중 경험(Multiexperience), 개인정보 보호 강화, 분산 클라우드, 투명성 및 추적성 강화 등 최신 ICT 트렌드와 개념들이 반영되어 있습니다.

요약하면, 이 문서는 서울시가 시민 중심의 첨단 ICT 인프라와 서비스를 구축하여 스마트도시를 구현하고, 안전하고 편리한 디지털 환경을 조성하기 위한 종합 계획과 전략을 담고 있습니다.


In [2]:
# =========================================================
# LangChain PDF RAG 필수 라이브러리 설치
# =========================================================

# 기존 코드
# !pip install langchain langchain-community langchain-openai PyMuPDF faiss-cpu nest_asyncio

# 개선 이유:
# - langchain-text-splitters 추가 (최신 분리 구조 필수)
# - requests 추가 (PDF URL 다운로드 안정성)
# - PyMuPDF 유지 (PDF parsing 성능 우수)

!pip install -q \
langchain \
langchain-community \
langchain-openai \
langchain-text-splitters \
faiss-cpu \
pypdf \
PyMuPDF \
nest_asyncio

In [24]:
"""
데이터흐름

PDF → Document → Chunk → Embedding → FAISS
→ Retriever → Document[]
→ format_docs → string
→ Prompt → LLM → Answer

"""

# =========================================================
# 2. 라이브러리 임포트
# =========================================================
# 역할: RAG 전체 파이프라인 구성 요소 불러오기

import os  # 환경변수(API KEY) 설정용
import nest_asyncio  # Colab/Jupyter async 충돌 방지
import requests  # PDF 다운로드 요청 처리

# LangChain PDF 로더 (PDF → Document 변환)
from langchain_community.document_loaders import PyMuPDFLoader

# 문서 분할 (Chunking)
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 벡터 DB (FAISS: 빠른 유사도 검색)
from langchain_community.vectorstores import FAISS

# LLM + Embedding 모델
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# Prompt + Output 처리
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# async 환경 충돌 방지




# =========================================================
# 3. API KEY 설정
# =========================================================
# 기존 방식: 하드코딩 (보안 위험)
# 개선 방식: Colab Secrets 사용

from google.colab import userdata

# OpenAI API Key 환경변수 등록
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")


# =========================================================
# 4. PDF 다운로드 (URL → 로컬 저장)
# =========================================================
pdf_url = "https://spri.kr/posts/view/23908"  # 웹페이지 URL
local_pdf_path = "/tmp/downloaded.pdf"       # Colab 내부 저장 경로

# HTTP 요청으로 파일 다운로드
response =


# 응답 상태 확인 (200 = 정상)
if response.status_code == 200:
    # 바이너리 파일 저장
    with open(local_pdf_path, "wb") as f:
        f.write(response.content)
else:
    # 실패 시 에러 발생
    raise Exception("PDF 다운로드 실패")

# 다운로드 완료 로그
print("PDF 저장 완료:", local_pdf_path)


# =========================================================
# 5. PDF 로딩
# =========================================================
# 역할: PDF 파일 → LangChain Document 객체로 변환

loader =   # PDF 로더 생성

docs =   # 페이지 단위 Document 리스트 생성


# =========================================================
# 6. Text Splitting (Chunking)
# =========================================================
# 역할: 긴 문서를 검색 가능한 단위로 나누기

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,   # 한 chunk 최대 길이 (문맥 유지)
    chunk_overlap=200  # 앞뒤 문맥 연결 (검색 품질 향상)
)

# Document → 작은 chunk 리스트로 변환
split_docs =



# =========================================================
# 7. Embedding + Vector DB 생성
# =========================================================
# 역할: 텍스트 → 벡터 변환 후 FAISS에 저장

embeddings =

# 의미 기반 벡터 생성 모델 (OpenAI embedding)

vectorstore = FAISS.from_documents(
    documents=split_docs,  # chunk 데이터
    embedding=embeddings   # embedding 모델
)


# =========================================================
# 8. Retriever 생성
# =========================================================
# 역할: 질문 → 관련 문서 검색 엔진

retriever =


# =========================================================
# 9. Context 변환 함수 (핵심 안정 포인트)
# =========================================================
def format_docs(docs):
    """
    역할:
    Document 리스트 → 문자열 변환

    이유:
    - LLM은 Document / dict 처리 불가
    - 반드시 string input 필요
    """
    return "\n\n".join(
        doc.page_content  # 각 문서 내용 추출
        for doc in docs   # 모든 chunk 순회
    )


# =========================================================
# 10. Prompt 정의 (LCEL 표준)
"""
너는 PDF 문서를 분석하는 AI이다.

아래 Context를 근거로 질문에 답변하라.

답변에는 Context의 내용을 요약하거나 인용하여 설명하라.

Context에 관련 내용이 전혀 없는 경우에만
"문서에 없음"이라고 답변한다.
"""
# =========================================================
prompt = ChatPromptTemplate.from_template(
"""
너는 PDF 기반 QA 시스템이다.

반드시 Context만 사용해서 답변해라.
모르면 "문서에 없음"이라고 답변해라.
추측 금지.

----------------------
Context:
{context}

Question:
{question}

----------------------
Answer:
"""
)


# =========================================================
# 11. LLM 정의
# =========================================================
llm = ChatOpenAI(
    model="gpt-4o-mini",  # 비용 대비 성능 최적 모델
    temperature=0         # 결정적 출력 (RAG 필수)
)


# =========================================================
# 12. LCEL Chain 구성 (핵심 실행 구조)
# =========================================================
# 역할:
# question → retriever → context 생성 → prompt → LLM → answer

chain = (
    {
        # 사용자 질문 그대로 전달
        "question":

        # retriever 실행 → Document[] → string 변환
        "context":

    }

    | prompt | llm | StrOutputParser()  # 결과 문자열 변환
)


# =========================================================
# 13. 실행 테스트
# =========================================================
question = "이 문서에서 가장 중요한 AI 트렌드는 무엇인가요?"

# chain 실행 (RAG 전체 흐름 실행)
result = chain.invoke({"question": question})

# 결과 출력
print("\n===== 답변 =====\n")
print(result)

PDF 저장 완료: /tmp/downloaded.pdf

===== 답변 =====

문서에 없음


네이버 뉴스기사 QA(Question-Answer)

이 가이드에서는 OpenAI 챗 모델과 임베딩, 그리고 Chroma 벡터 스토어를 사용할 것입니다.

먼저 다음의 과정을 통해 간단한 인덱싱 파이프라인과 RAG 체인을 약 20줄의 코드로 구현할 수 있습니다.

라이브러리

- bs4는 웹 페이지를 파싱하기 위한 라이브러리입니다.
- langchain은 AI와 관련된 다양한 기능을 제공하는 라이브러리로, 여기서는 특히
  1. 텍스트 분할(RecursiveCharacterTextSplitter),
  2. 문서 로딩(WebBaseLoader),
  3. 벡터 저장(Chroma, FAISS),
  4. 출력 파싱(StrOutputParser),
  5. 실행 가능한 패스스루(RunnablePassthrough) 등을 다룹니다.
- langchain_openai 모듈을 통해 OpenAI의 챗봇(ChatOpenAI)과 임베딩(OpenAIEmbeddings) 기능을 사용할 수 있습니다.

In [ ]:

# =========================================================
# 1. 라이브러리 설치
# =========================================================
# 역할: RAG / 웹 크롤링 / 벡터DB / LCEL 실행 환경 구성
# - LCEL + Web RAG 구조 필수 구성 유지
# - FAISS 기반 실시간 검색 구조

!pip install -q langchain langchain-community langchain-openai langchainhub faiss-cpu beautifulsoup4 nest_asyncio


LLM에 들어가는 입력을 통제해서 RAG 결과를 안정화한 코드
- Document → string 변환을 강제한 안정형 LCEL RAG 체인
- Retriever가 반환한 Document를 LLM이 처리 가능한 문자열로 변환해 LCEL 파이프라인에서 데이터 타입 안정성과 예측 가능한 RAG 동작을 보장한 구조

In [25]:

# =========================================================
# 2. 라이브러리 임포트
# =========================================================
# 역할: RAG 전체 파이프라인 구성 요소 로딩

import os
import bs4
import nest_asyncio

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from langchain_core.runnables import RunnableMap

nest_asyncio.apply()
# Colab/Jupyter async 충돌 방지


# =========================================================
# 3. API KEY 설정
# =========================================================
# 기존: 하드코딩
# 개선: Colab Secrets

from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")


# =========================================================
# 4. BS4 필터 설정 (뉴스 본문만 추출)
# =========================================================
# 역할: HTML → 필요한 본문만 필터링

soup_filter = bs4.SoupStrainer(
    "div",
    attrs={"class": ["newsct_article _article_body", "media_end_head_title"]}
)


# =========================================================
# 5. 웹 문서 로딩 (네이버 뉴스)
# =========================================================
url = "https://n.news.naver.com/article/437/0000378416"

loader = WebBaseLoader(


)

docs = loader.load()
print(f"문서 로딩 완료: {len(docs)}개")


# =========================================================
# 6. Chunking (문서 분할)
# =========================================================
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,   # 의미 유지 단위
    chunk_overlap=200  # 문맥 연결 유지
)

splits =

print(f"청크 개수: {len(splits)}")


# =========================================================
# 7. Vector DB (FAISS)
# =========================================================
embeddings =


vectorstore =


retriever =


# =========================================================
# 8. Context 변환 함수 (핵심 안정성 포인트)
# =========================================================
def format_docs(docs):
    """
    Document → string 변환
    이유:
    - LLM은 Document 타입을 직접 처리 못함
    - 반드시 문자열 context 필요
    """
    return "\n\n".join(d.page_content for d in docs)


# =========================================================
# 9. Prompt 설계 (LCEL 표준)
# =========================================================
prompt = ChatPromptTemplate.from_template("""
너는 뉴스 기사 QA 시스템이다.

반드시 Context만 기반으로 답변해라.
문서에 없는 내용은 추측하지 말고 "문서에 없음"이라고 답변해라.

----------------------
Context:
{context}

Question:
{question}

----------------------
Answer:
""")


# =========================================================
# 10. LLM 정의
# =========================================================
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)


# =========================================================
# 11. LCEL Chain (정석 구조)
# =========================================================
# 핵심 흐름:
# question → retriever → format_docs → prompt → LLM → output

rag_chain = (
    {
        # 질문 전달
        "question":


        # retriever 실행 → Document[] → string 변환
        "context":

    }
    | prompt
    | llm
    | StrOutputParser()
)


# =========================================================
# 12. 실행 함수
# =========================================================
def ask(question):
    """
    RAG QA 실행 함수
    """
    print("\n====================")
    print("질문:", question)
    print("====================")

    result = rag_chain.invoke({"question": question})

    print("\n답변:\n", result)


# =========================================================
# 13. 테스트 실행
# =========================================================
ask("부영그룹의 출산 장려 정책은 무엇인가요?")
ask("정부의 저출생 대책을 bullet point로 정리해 주세요")

문서 로딩 완료: 1개
청크 개수: 3

질문: 부영그룹의 출산 장려 정책은 무엇인가요?

답변:
 부영그룹의 출산 장려 정책은 2021년 이후 태어난 직원 자녀에게 1억원씩 지원하는 것으로, 총 70억원을 지원할 계획입니다. 연년생이나 쌍둥이 자녀가 있는 경우에는 총 2억원을 받을 수 있으며, 셋째까지 낳는 경우에는 국민주택을 제공하겠다는 뜻도 밝혔습니다.

질문: 정부의 저출생 대책을 bullet point로 정리해 주세요

답변:
 - 매달 주는 부모 급여: 0세 아이는 100만원
- 첫만남이용권 및 아동수당 포함 시, 아이 돌까지 1년 동안 총 1520만원 지원
- 인천시: 새로 태어난 아기에게 18살까지 1억원 지원
- 광주시: 17살까지 7400만원 지원
- 출산율 저하에 따른 현금성 지원 정책화 추진


LangChain 기반 RAG(Retrieval-Augmented Generation) 워크플로우
1. 인덱싱 단계 (Indexing - Offline)
2. 검색 + 생성 단계 (Retrieval + Generation - Online)

RAG 의 기능별 다양한 모듈 활용기

- 인덱싱: 소스에서 데이터를 수집하고 인덱싱하는 파이프라인입니다. 이 작업은 보통 오프라인에서 발생합니다.

- 검색 및 생성: 실제 RAG 체인으로, 사용자 쿼리를 실행 시간에 받아 인덱스에서 관련 데이터를 검색한 다음, 그 데이터를 모델에 전달합니다.

- RAW 데이터에서 답변을 받기까지의 전체 순서는 다음과 같습니다.

  1. 인덱싱
    - 로드: 먼저 데이터를 로드해야 합니다. 이를 위해 DocumentLoaders를 사용할 것입니다.
    - 분할: Text splitters는 큰 Documents를 더 작은 청크로 나눕니다. 이는 데이터를 인덱싱하고 모델에 전달하는 데 유용하며, 큰 청크는 검색하기 어렵고 모델의 유한한 컨텍스트 창에 맞지 않습니다.
    - 저장: 나중에 검색할 수 있도록 분할을 저장하고 인덱싱할 장소가 필요합니다. 이는 종종 VectorStore와 Embeddings 모델을 사용하여 수행됩니다.

  2. 검색 및 생성
    - 검색: 사용자 입력이 주어지면 Retriever를 사용하여 저장소에서 관련 분할을 검색합니다.
    - 생성: ChatModel / LLM은 질문과 검색된 데이터를 포함한 프롬프트를 사용하여 답변을 생성합니다

In [ ]:
"""
전체 RAG 파이프라인


[INDEXING]
Document
→ Split
→ Embedding
→ Vector DB

[QUERY TIME]
User Question
→ Retriever (Vector Search)
→ Context Documents
→ Prompt
→ LLM
→ Answer

"""

전체 기능 요약

이 코드는 다음과 같은 순서로 동작하는 최신 LCEL 기반 RAG(Retrieval-Augmented Generation) 시스템입니다.

1. 웹 페이지(또는 PDF 등)를 LangChain Document 객체로 불러옵니다.

2. 긴 문서를 RecursiveCharacterTextSplitter를 이용하여
검색하기 적합한 크기의 Chunk로 분할합니다.

3. 각 Chunk를 OpenAI Embedding 모델을 이용하여
의미(Vector)로 변환합니다.

4. 생성된 Embedding을 FAISS Vector Database에 저장하여
의미 기반(Semantic Search) 검색이 가능하도록 합니다.

5. 사용자의 질문(Query)을 Embedding으로 변환한 후
FAISS Retriever가 가장 관련성이 높은 Chunk를 검색합니다.

6. 검색된 Context와 사용자 질문을 Prompt로 구성하여
ChatOpenAI가 최종 답변을 생성합니다.

7. StrOutputParser를 이용하여 LLM의 응답을 문자열(String) 형태로 반환합니다.

In [ ]:
"""

User Question
      │
      ▼
WebBaseLoader
      │
      ▼
RecursiveCharacterTextSplitter
      │
      ▼
OpenAIEmbeddings
      │
      ▼
FAISS
      │
      ▼
Retriever
      │
      ▼
RunnableParallel
      │
      ├────────► Question
      │
      └────────► Context
                   │
                   ▼
              format_docs()
                   │
                   ▼
          ChatPromptTemplate
                   │
                   ▼
             ChatOpenAI
                   │
                   ▼
          StrOutputParser
                   │
                   ▼
              Final Answer

"""

In [11]:
# =========================================================
# 2. OpenAI / 검색 / 벡터DB / 파싱 관련
# =========================================================
# 역할:
# - OpenAI LLM + Embedding
# - FAISS 벡터 검색
# - BM25 키워드 검색
# - HTML parsing

!pip install -q \
    openai \
    faiss-cpu \
    rank_bm25 \
    beautifulsoup4 \
    tiktoken \
    bs4

In [12]:
# =========================================================
# 2. OpenAI / 검색 / 벡터DB / 파싱 관련
# =========================================================
# 역할:
# - OpenAI LLM + Embedding
# - FAISS 벡터 검색
# - BM25 키워드 검색
# - HTML parsing

!pip install -q \
    openai \
    faiss-cpu \
    rank_bm25 \
    beautifulsoup4 \
    tiktoken \
    bs4

In [15]:
# ============================================================
# 실행 흐름 (Execution Flow)
# ============================================================
#
# 사용자 질문(User Question)
#          │
#          ▼
#      WebBaseLoader
#          │
#          ▼
#   HTML → Document 변환
#          │
#          ▼
# RecursiveCharacterTextSplitter
#          │
#          ▼
#      Chunk 생성
#          │
#          ▼
# Part 2
# (Embedding + FAISS Retriever)
#
# ============================================================


# ============================================================
# 1. API KEY 설정
# ============================================================
#
# 역할
# ------------------------------------------------------------
# OpenAI API를 사용하기 위해
# OPENAI_API_KEY를 환경 변수에 등록한다.
#
# Google Colab Secret에 저장된
# API KEY를 읽어온다.
# ============================================================

import os
from google.colab import userdata

# Colab Secret → 환경 변수 등록
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")


# ============================================================
# 2. 라이브러리 Import
# ============================================================
#
# 역할
# ------------------------------------------------------------
# 최신 LangChain 1.x에서 사용하는
# RAG 구성 요소를 불러온다.
#
# 주요 구성
#
# - WebBaseLoader
# - RecursiveCharacterTextSplitter
# - FAISS
# - OpenAIEmbeddings
# - ChatOpenAI
# - LCEL Runnable
# ============================================================

# HTML Parsing
import bs4

# 웹 문서를 Document 객체로 변환
from langchain_community.document_loaders import WebBaseLoader

# 최신 Text Splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Vector Database
from langchain_community.vectorstores import FAISS

# OpenAI Embedding + Chat Model
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

# Prompt Template
from langchain_core.prompts import ChatPromptTemplate

# Output Parser
from langchain_core.output_parsers import StrOutputParser

# LCEL Runnable
from langchain_core.runnables import (
    RunnableLambda,
    RunnableParallel,
    RunnablePassthrough,
)


# ============================================================
# 3. 뉴스 기사 로딩
# ============================================================
#
# 역할
# ------------------------------------------------------------
# WebBaseLoader를 이용하여
# 네이버 뉴스 기사를 Document 객체로 변환한다.
#
# BeautifulSoup 필터를 이용하여
#
# - 기사 제목
# - 기사 본문
#
# 만 가져온다.
# ============================================================

# 뉴스 기사 URL
url = "https://n.news.naver.com/article/437/0000378416"

# HTML 필터
soup_filter = bs4.SoupStrainer(
    "div",
    attrs={
        "class": [
            "media_end_head_title",        # 기사 제목
            "newsct_article _article_body" # 기사 본문
        ]
    },
)

# Web Loader 생성
loader = WebBaseLoader(


)

# 웹 문서를 Document 객체로 변환
docs = loader.load()

# 결과 확인
print("=" * 60)
print("문서 개수 :", len(docs))
print("=" * 60)
print("첫 번째 문서 일부")
print(docs[0].page_content[:500])


# ============================================================
# 4. Text Splitter
# ============================================================
#
# 역할
# ------------------------------------------------------------
# Retriever는 긴 문서를 그대로 검색하지 않는다.
#
# 검색 성능 향상을 위해
# 문서를 작은 Chunk로 분할한다.
#
# RecursiveCharacterTextSplitter는
# 의미(Context)를 최대한 유지하면서
# 문서를 분할하는 RAG 표준 Splitter이다.
# ============================================================

# Text Splitter 생성
text_splitter = RecursiveCharacterTextSplitter(
    # 하나의 Chunk 최대 크기
    # 문맥(Context) 유지
    # 문자열 길이 기준
)

# Document → Chunk(Document)
split_docs =

# 결과 확인
print("=" * 60)
print("생성된 Chunk 개수 :", len(split_docs))
print("=" * 60)
print("첫 번째 Chunk")
print(split_docs[0].page_content[:500])

# ============================================================
# 5. Embedding Model 생성
# ============================================================
#
# 역할
# ------------------------------------------------------------
# Embedding Model은 텍스트를 숫자(Vector)로 변환한다.
#
# 의미(Semantic)를 수치화하여
# 질문과 문서 간의 유사도를 계산할 수 있게 한다.
#
# RAG에서는 Retriever가 의미 기반 검색을 수행하기 위해
# 반드시 Embedding Model이 필요하다.
#
# 여기서는 OpenAI의
#
# text-embedding-3-small
#
# 모델을 사용한다.
# ============================================================

# Embedding 모델 생성
embedding_model =



# ============================================================
# 6. FAISS Vector Store 생성
# ============================================================
#
# 역할
# ------------------------------------------------------------
# 분할된 Chunk(Document)를
#
# ① Embedding(Vector)으로 변환하고
# ② FAISS Vector Database에 저장한다.
#
# 이후 Retriever는
# FAISS에 저장된 Vector를 검색하여
# 질문과 가장 유사한 Chunk를 찾는다.
# ============================================================

# Chunk → Embedding → FAISS 저장
vectorstore =


# 저장된 Vector 개수 확인
print("=" * 60)
print("FAISS Vector 개수 :", vectorstore.index.ntotal)


# ============================================================
# 7. Retriever 생성
# ============================================================
#
# 역할
# ------------------------------------------------------------
# Retriever는
# 사용자의 질문(Query)을 입력받아
# 가장 관련성이 높은 Chunk를 검색한다.
#
# Retriever는
#
# 질문
#      │
#      ▼
# Embedding 생성
#      │
#      ▼
# Vector Search
#      │
#      ▼
# Top-k Document 반환
#
# 과정을 수행한다.
#
# search_kwargs
#
# k
# → 검색할 Chunk 개수
# ============================================================

# Retriever 생성
retriever =



# ============================================================
# 8. Retriever 동작 테스트
# ============================================================
#
# 역할
# ------------------------------------------------------------
# Retriever가 실제로
# 관련 문서를 검색하는지 확인한다.
#
# invoke()의 반환값은
#
# List[Document]
#
# 이다.
# ============================================================

# 검색 테스트
test_docs = retriever.invoke(
    "부영그룹의 출산 장려 정책은 무엇인가요?"
)

# 검색 결과 확인
print("=" * 60)
print("검색된 문서 개수 :", len(test_docs))
print("=" * 60)

# 검색된 문서 출력
for idx, doc in enumerate(test_docs, start=1):

    print(f"\n===== Document {idx} =====\n")

    print(doc.page_content[:500])


# ============================================================
# 9. Retriever 반환 타입 확인
# ============================================================
#
# 역할
# ------------------------------------------------------------
# Retriever는
#
# List[Document]
#
# 를 반환한다.
#
# Prompt는 문자열(String)을 입력으로 사용하므로
# 다음 단계에서는
#
# Document
#      │
#      ▼
# String(Context)
#
# 으로 변환해야 한다.
# ============================================================

print("=" * 60)
print("Retriever 반환 타입")
print(type(test_docs))
print(type(test_docs[0]))

# ============================================================
# 10. Context 변환 함수
# ============================================================
#
# 역할
# ------------------------------------------------------------
# Retriever는 검색 결과를
#
# List[Document]
#
# 형태로 반환한다.
#
# 하지만 Prompt는 문자열(String)을 입력으로 사용한다.
#
# 따라서 Document 객체를 하나의 문자열(Context)로
# 변환하는 과정이 반드시 필요하다.
#
# 변환 과정
#
# List[Document]
#          │
#          ▼
# format_docs()
#          │
#          ▼
# 하나의 문자열(Context)
#
# ============================================================

def format_docs(docs):
    """
    Document 리스트를 하나의 문자열(Context)로 변환한다.
    """

    return "\n\n".join(
        doc.page_content
        for doc in docs
    )


# ============================================================
# 11. Prompt Template 생성
# ============================================================
#
# 역할
# ------------------------------------------------------------
# Prompt는
#
# Retriever가 검색한 Context와
# 사용자의 Question을 하나의 Prompt로 만든다.
#
# Prompt 구조
#
# Context
# +
# Question
# ↓
# LLM
#
# ============================================================

prompt = ChatPromptTemplate.from_template(
"""
당신은 뉴스 기사 기반 QA 시스템입니다.

반드시 아래 Context만 참고하여 답변하세요.

Context에 없는 내용은
"문서에 없음"이라고 답변하세요.

추측하거나 임의로 생성하지 마세요.

------------------------------------------------
Context
{context}
------------------------------------------------
Question
{question}
------------------------------------------------
Answer
"""
)


# ============================================================
# 12. Chat Model 생성
# ============================================================
#
# 역할
# ------------------------------------------------------------
# ChatOpenAI는
# Prompt를 입력받아 최종 답변을 생성한다.
#
# temperature = 0
#
# → 항상 일관된 답변 생성
# ============================================================

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)


# ============================================================
# 13. LCEL RAG Chain 구성
# ============================================================
#
# 실행 흐름
#
# 사용자 질문
#      │
#      ▼
# RunnableParallel
#      │
#      ├────────────► question
#      │
#      └────────────► Retriever
#                        │
#                        ▼
#                  format_docs()
#                        │
#                        ▼
#                     Context
#                        │
#                        ▼
#                Prompt Template
#                        │
#                        ▼
#                  ChatOpenAI
#                        │
#                        ▼
#               StrOutputParser
#                        │
#                        ▼
#                  최종 문자열 출력
#
# ============================================================

rag_chain = (

    RunnableParallel(

        # ----------------------------------------------------
        # 사용자 질문 그대로 전달
        # ----------------------------------------------------
        question=

        # ----------------------------------------------------
        # Retriever 실행
        #
        # Question
        #      │
        #      ▼
        # Retriever
        #      │
        #      ▼
        # List[Document]
        #      │
        #      ▼
        # format_docs()
        #      │
        #      ▼
        # Context(String)
        # ----------------------------------------------------
        context=(


        ),
    ) | prompt | llm | StrOutputParser()
)

# ============================================================
# 14. LCEL Chain 구조 확인
# ============================================================
#
# 역할
# ------------------------------------------------------------
# 생성된 LCEL Chain의 실행 구조를 확인한다.
#
# 학습 및 디버깅 시 매우 유용하다.
# ============================================================

print("=" * 60)
print("LCEL Chain 생성 완료")
print("=" * 60)

print(rag_chain)


문서 개수 : 1
첫 번째 문서 일부

출산 직원에게 '1억원' 쏜다…회사의 파격적 저출생 정책


[앵커]올해 아이 낳을 계획이 있는 가족이라면 솔깃할 소식입니다. 정부가 저출생 대책으로 매달 주는 부모 급여, 0세 아이는 100만원으로 올렸습니다. 여기에 첫만남이용권, 아동수당까지 더하면 아이 돌까지 1년 동안 1520만원을 받습니다. 지자체도 경쟁하듯 지원에 나섰습니다. 인천시는 새로 태어난 아기, 18살될 때까지 1억원을 주겠다. 광주시도 17살될 때까지 7400만원 주겠다고 했습니다. 선거 때면 나타나서 아이 낳으면 현금 주겠다고 밝힌 사람이 있었죠. 과거에는 표만 노린 '황당 공약'이라는 비판이 따라다녔습니다. 그런데 지금은 출산율이 이보다 더 나쁠 수 없다보니, 이런 현금성 지원을 진지하게 정책화 하는 상황까지 온 겁니다. 게다가 기업들도 뛰어들고 있습니다. 이번에는 출산한 직원에게 단번에 1억원을 주겠다는 회사까지 나타났습니다.이상화 기자가 취재했습니다.[기자]한 그룹사가 오늘 파격적인 저출생 정책을 내놨
생성된 Chunk 개수 : 3
첫 번째 Chunk
출산 직원에게 '1억원' 쏜다…회사의 파격적 저출생 정책
FAISS Vector 개수 : 3
검색된 문서 개수 : 3

===== Document 1 =====

출산 직원에게 '1억원' 쏜다…회사의 파격적 저출생 정책

===== Document 2 =====

부정적이었거든요. (이제) 긍정적으로 생각할 수 있을 것 같습니다.]오늘 행사에서는, 회사가 제공하는 출산장려금은 받는 직원들의 세금 부담을 고려해 정부가 면세해달라는 제안도 나왔습니다.이같은 출산장려책은 점점 확산하는 분위기입니다.법정기간보다 육아휴직을 길게 주거나, 남성 직원의 육아휴직을 의무화한 곳도 있습니다.사내 어린이집을 밤 10시까지 운영하고 셋째를 낳으면 무조건 승진시켜 주기도 합니다.한 회사는 지난해 네쌍둥이를 낳은 직원에 의료비를 지원해 관심을 모았습니다.정부 대신 회사가 나서는 출산장려책이 사회적 분위기를 바꿀 거라는

# 개선된 실무용 RAG
# 개선 RAG 구조 (Chat Memory + Multi Query RAG)

기존 RAG 시스템은 사용자의 질문을 한 번만 검색하여 관련 문서를 찾고, 검색된 문서를 기반으로 LLM이 답변을 생성하는 구조였다. 하지만 질문 표현이 조금만 달라져도 검색 성능이 저하되고, 이전 대화 내용을 기억하지 못하는 한계가 있었다.

이를 개선하기 위해 **Chat Memory**와 **Multi Query RAG**를 적용하여 검색 정확도와 대화 연속성을 향상시켰다.

---

## 1. 검색 정확도 개선 (Multi Query RAG)

### 기존 방식
- 하나의 질문으로 한 번만 검색 수행
- 질문 표현이 달라질 경우 검색 성능 저하

### 개선 방식
- 하나의 질문을 여러 개의 유사 질문으로 자동 생성
- 생성된 모든 질문으로 문서를 검색
- 검색 결과를 통합하여 LLM에 전달

### 기대 효과
- 다양한 표현에 대한 검색 성능 향상
- 관련 문서 검색 누락 감소
- Recall(재현율) 향상

---

## 2. 대화 기억 기능 (Chat Memory)

### 기존 방식
- 질문마다 독립적으로 처리
- 이전 대화 내용을 고려하지 않음

### 개선 방식
- 이전 질문과 답변을 함께 저장
- 새로운 질문과 함께 이전 대화 내용을 Context로 활용

### 기대 효과
- 연속적인 질의응답 가능
- 문맥(Context) 유지
- 자연스러운 대화형 RAG 구현

---

## 3. LCEL 기반 파이프라인 유지

기존 LCEL(Runnable) 구조를 그대로 유지하면서 기능을 확장하였다.

```
User Question
      │
      ▼
Chat Memory
      │
      ▼
Multi Query 생성
      │
      ▼
Retriever
      │
      ▼
format_docs()
      │
      ▼
ChatPromptTemplate
      │
      ▼
ChatOpenAI
      │
      ▼
StrOutputParser
      │
      ▼
Final Answer
```

---

## 4. Context 생성 방식 개선

검색된 Document 객체는 LLM이 직접 처리하기 어렵기 때문에 `format_docs()` 함수를 사용하여 하나의 문자열(Context)로 변환한다.

이를 통해

- LLM 입력 형식을 통일하고
- Document 타입 오류를 방지하며
- 안정적인 RAG 파이프라인을 구성할 수 있다.

---

## 5. 개선된 RAG의 특징

- Multi Query를 통한 검색 성능 향상
- Chat Memory를 활용한 대화 연속성 유지
- LCEL 기반의 모듈화된 파이프라인 구성
- 안정적인 Context 생성
- 향후 LangGraph Agent, Tool Calling, Context Compression 등으로 쉽게 확장 가능한 구조

---

## 전체 기능 요약

개선된 RAG 시스템은 다음과 같은 순서로 동작한다.

1. 사용자의 질문을 입력받는다.
2. 이전 대화(Chat Memory)를 함께 고려한다.
3. 질문을 여러 개의 검색 질의(Multi Query)로 확장한다.
4. 각 질의를 이용하여 관련 문서를 검색한다.
5. 검색된 문서를 하나의 Context로 구성한다.
6. Prompt Template에 Context와 질문을 전달한다.
7. ChatOpenAI가 최종 답변을 생성한다.
8. StrOutputParser를 통해 문자열 형태의 최종 결과를 반환한다.

---

## 핵심 정리

기존 RAG가 **"한 번의 검색을 수행하는 문서 기반 질의응답 시스템"** 이었다면,

개선된 RAG는 **"대화 기억(Chat Memory)과 질의 확장(Multi Query)을 결합하여 검색 정확도와 대화 연속성을 향상시킨 실무형 RAG 시스템"**이다.

In [ ]:
# ================================================
# 1. 필요한 라이브러리 설치 (Colab 환경)
# ================================================
# 역할:
# - LangChain RAG 전체 구성 (Loader / Retriever / Chain)
# - OpenAI LLM + Embedding
# - FAISS 벡터 검색
# - BM25 키워드 검색
# - HTML 파싱 및 토큰 처리

!pip install -q \
    langchain \
    langchain-community \
    langchain-openai \
    langchain-experimental \
    openai \
    faiss-cpu \
    tiktoken \
    beautifulsoup4

In [19]:

# ============================================================
# 1. IMPORTS
# ============================================================

from langchain_openai import ChatOpenAI, OpenAIEmbeddings  # OpenAI LLM + Embedding 모델
from langchain_community.vectorstores import FAISS  # 벡터DB (FAISS)
from langchain_core.prompts import ChatPromptTemplate  # Prompt 템플릿
from langchain_core.output_parsers import StrOutputParser  # 문자열 출력 변환
from langchain_core.runnables import RunnableParallel  # 병렬 실행 (question/context 분리)
from langchain_core.runnables.history import RunnableWithMessageHistory  # Memory 연결
from langchain_community.chat_message_histories import ChatMessageHistory  # 대화 저장소


# ============================================================
# 2. LLM 설정
# ============================================================

llm = ChatOpenAI(
    model="gpt-4o-mini",   # 경량 GPT 모델
    temperature=0          # 일관된 답변 (RAG는 0이 안정적)
)


# ============================================================
# 3. SAMPLE DOCUMENTS (실제 환경에서는 외부 데이터로 교체)
# ============================================================

docs = [
    "부영그룹은 출산 장려 정책으로 자녀 1명당 1억 원 지원을 발표했다.",
    "이 정책은 저출산 문제 해결과 인구 감소 대응을 목표로 한다.",
    "기업 복지와 사회적 책임 강화 목적도 포함된다."
]


# ============================================================
# 4. VECTOR STORE (FAISS)
# ============================================================

embeddings = OpenAIEmbeddings()  # 문서를 벡터로 변환하는 모델

vectorstore = FAISS.from_texts(
    docs,          # 문서 리스트
    embeddings     # 임베딩 모델
)

retriever = vectorstore.as_retriever()  # 검색기 (Query → 관련 문서 반환)


# ============================================================
# 5. MEMORY (Chat History)
# ============================================================

chat_history =

# 세션별 메모리 관리 함수
def get_session_history(session_id: str):
    """
    역할:
    - session_id 기반으로 대화 기록 반환
    - 현재는 단일 사용자 구조
    """
    return

session_id = "user_1"  # 테스트용 세션 ID


# ============================================================
# 6. MULTI QUERY GENERATION
# ============================================================

multi_query_prompt = ChatPromptTemplate.from_template(
"""
다음 질문을 3개의 서로 다른 검색 질문으로 변환해라.

질문:
{question}

출력:
1.
2.
3.
"""
)

# LLM → 3개의 검색 query 생성

multi_query_chain =


# ============================================================
# 7. RETRIEVAL FUNCTION (Multi Query 기반 검색)
# ============================================================

def retrieve_multi_query(question):
    """
    역할:
    - 질문을 3개로 확장
    - 각각 FAISS 검색 수행
    - 결과 문서 합치기
    """

    # 1) 질문 확장 (3개 query 생성)
    queries =


    # 2) 문자열 → 리스트 변환
    query_list = [

    ]

    docs = []  # 결과 문서 저장

    # 3) 각 query별 retrieval 수행
    for q in query_list:



    return docs


# ============================================================
# 8. DOCUMENT FORMATTER
# ============================================================

def format_docs(docs):
    """
    역할:
    - 검색된 Document 리스트 → 문자열 변환
    - LLM context로 전달하기 위한 전처리
    """
    return "\n\n".join(d.page_content for d in docs)


# ============================================================
# 9. PROMPT (Memory + RAG 통합)
# ============================================================

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
당신은 RAG 기반 QA 시스템입니다.

규칙:
- 반드시 Context만 사용
- 모르면 "문서에 없음"이라고 답변
"""
    ),

    # Memory (이전 대화 삽입 위치)



    (
        "human",
        """
Context:
{context}

Question:
{question}
"""
    )
])


# ============================================================
# 10. LCEL RAG CHAIN
# ============================================================

rag_chain = (
    RunnableParallel(
        # 질문 그대로 전달
        question=lambda x: x["question"],

        # context 생성 (multi-query retrieval)
        context=lambda x: format_docs(
            retrieve_multi_query(x["question"])
        )
    )
    | prompt              # prompt에 question/context/history 결합
    | llm                 # LLM 호출
    | StrOutputParser()   # 문자열 출력 변환
)


# ============================================================
# 11. MEMORY WRAPPER
# ============================================================

memory_chain = RunnableWithMessageHistory(
    runnable=rag_chain,                 # 기존 RAG chain
    get_session_history=get_session_history,  # memory provider
    input_messages_key="question",     # 입력 key
    history_messages_key="history"      # prompt 내 memory slot
)


# ============================================================
# 12. EXECUTION FUNCTION
# ============================================================

def ask(question):
    """
    역할:
    - 사용자 질문 입력
    - Memory + RAG chain 실행
    - 결과 출력
    """

    print("\n" + "="*60)
    print("QUESTION:", question)
    print("="*60)

    result = memory_chain.invoke(
        {"question": question},
        config={"configurable": {"session_id": session_id}}
    )

    print("\nANSWER:\n", result)


# ============================================================
# 13. TEST RUN
# ============================================================

ask("부영그룹의 출산 장려 정책은 무엇인가요?")
ask("그 정책의 목적을 다시 설명해줘")


QUESTION: 부영그룹의 출산 장려 정책은 무엇인가요?

ANSWER:
 부영그룹의 출산 장려 정책은 자녀 1명당 1억 원 지원을 발표한 것으로, 저출산 문제 해결과 인구 감소 대응을 목표로 하고 있습니다. 또한 기업 복지와 사회적 책임 강화 목적도 포함됩니다.

QUESTION: 그 정책의 목적을 다시 설명해줘

ANSWER:
 그 정책의 목적은 저출산 문제 해결과 인구 감소 대응이며, 기업 복지와 사회적 책임 강화도 포함됩니다.


# 이전 대화를 기억하는 Chain 생성방법


## 1. 일반 Chain 에 대화기록 추가

In [ ]:

# =========================================================
# 1. LangChain RAG 핵심 패키지 설치
# =========================================================
# 역할:
# - LLM 기반 RAG 전체 구성 (Chain / Retriever / Splitter)
# - OpenAI 연동
# - 문서 처리 + 벡터 검색

!pip install -q \
    langchain \
    langchain-openai \
    langchain-community \
    langchain-text-splitters


# =========================================================
# 2. 벡터 DB / 토크나이저 / 환경관리
# =========================================================
# 역할:
# - FAISS: 벡터 검색 엔진 (Dense retrieval)
# - tiktoken: OpenAI 토큰 계산
# - dotenv: 로컬 환경 변수 관리

!pip install -q \
    faiss-cpu \
    tiktoken \
    python-dotenv

In [2]:
import os
from google.colab import userdata

os.environ['OPENAI_API_KEY'] = userdata.get("OPENAI_API_KEY")

In [3]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser


# =========================================================
# Prompt 정의 (Memory 포함)
# =========================================================
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "당신은 Question-Answering 챗봇입니다. 주어진 질문에 대한 답변을 제공해주세요.",
        ),



        ,
        ("human", "#Question:\n{question}"),
    ]
)


# =========================================================
# LLM (최신 권장 모델 명시)
# =========================================================
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)


# =========================================================
# LCEL Chain (변경 없음 - 최신 표준 구조)
# =========================================================
chain = prompt | llm | StrOutputParser()

대화를 기록하는 체인 생성(`chain_with_history`)

In [4]:
# =========================================================
# 세션 저장소 (in-memory)
# =========================================================
store = {}


# =========================================================
# 세션 히스토리 조회 함수
# =========================================================
def get_session_history(session_id: str):
    print(f"[대화 세션ID]: {session_id}")

    # 해당 session이 없으면 새로 생성
    if session_id not in store:
        store[session_id] =

    return


# =========================================================
# Memory가 포함된 LCEL Chain
# =========================================================
chain_with_history =



첫 번째 질문 실행

In [5]:

"""
=========================================================
전체 구조 (LCEL Memory Flow)
=========================================================

User Question
   ↓
session_id (abc123)
   ↓
RunnableWithMessageHistory
   ↓
ChatMessageHistory (store)
   ↓
Prompt (chat_history + question)
   ↓
LLM
   ↓
Answer + Memory 저장
=========================================================
"""


# =========================================================
# 실행 (Memory + LCEL Chain)
# =========================================================

chain_with_history.invoke(
    {
        "question": "나의 이름은 홍길동입니다."
    },
    config={
        "configurable": {
            "session_id": "abc123"
        }
    }
)

[대화 세션ID]: abc123


'안녕하세요, 홍길동님! 어떻게 도와드릴까요?'

이어서 질문 실행

In [6]:
chain_with_history.invoke(
    # 질문 입력
    {"question": "내 이름이 뭐라고?"},
    # 세션 ID 기준으로 대화를 기록합니다.
    config={"configurable": {"session_id": "abc123"}},
)

[대화 세션ID]: abc123


'당신의 이름은 홍길동입니다.'

## 2. RAG + RunnableWithMessageHistory

먼저 일반 RAG Chain 을 생성합니다. 단, 6단계의 prompt 에 `{chat_history}` 를 꼭 추가합니다.

In [7]:
!pip install -q pdfplumber

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 698.4 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 66.1 MB/s eta 0:00:00


In [18]:
# =========================================================
# 1. Imports (필요 라이브러리 로드)
# =========================================================

from google.colab import files  # PDF 파일 업로드 (Colab 전용)

from langchain_text_splitters import RecursiveCharacterTextSplitter  # 문서 chunk 분할
from langchain_community.document_loaders import PDFPlumberLoader  # PDF 텍스트 로더

from langchain_community.vectorstores import FAISS  # 벡터 DB (Dense retrieval)

from langchain_openai import ChatOpenAI, OpenAIEmbeddings  # LLM + Embedding 모델

from langchain_core.prompts import PromptTemplate  # Prompt 템플릿 생성
from langchain_core.output_parsers import StrOutputParser  # LLM 출력 문자열 변환

from langchain_core.runnables import RunnableParallel  # 병렬 실행 LCEL

from langchain_community.chat_message_histories import ChatMessageHistory  # memory 저장 객체
from langchain_core.runnables.history import RunnableWithMessageHistory  # session memory wrapper


# =========================================================
# 2. PDF Upload (사용자 파일 업로드)
# =========================================================

uploaded = files.upload()  # PDF 업로드 실행
pdf_file_path = next(iter(uploaded))  # 업로드된 파일 이름 추출


# =========================================================
# 3. PDF Load (문서 로딩)
# =========================================================

loader = PDFPlumberLoader(pdf_file_path)  # PDF loader 생성
docs = loader.load()  # PDF → Document 객체 변환


# =========================================================
# 4. Chunking (문서 분할)
# =========================================================

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # chunk 최대 길이
    chunk_overlap=50   # 문맥 유지용 overlap
)

split_docs = text_splitter.split_documents(docs)  # 문서 → chunk 리스트


# =========================================================
# 5. Embedding + Vector DB 생성
# =========================================================

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")  # embedding 모델

vectorstore = FAISS.from_documents(split_docs, embeddings)  # FAISS index 생성

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})  # top-k 검색 설정


# =========================================================
# 6. Prompt 정의 (RAG 입력 구조)
# =========================================================

prompt = PromptTemplate.from_template(
"""
You are a RAG assistant.  # 역할 정의 (LLM behavior control)




Question:  # user input
{question}




Answer:  # final output
"""
)


# =========================================================
# 7. LLM 설정
# =========================================================

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)  # deterministic output


"""
ARCHITECTURE NOTE (핵심 구조 설명)

RunnableWithMessageHistory
   ↓
ChatMessageHistory 자동 관리 (session 기반)
   ↓
session_id 기준으로 memory 분리
   ↓
MessagesPlaceholder(chat_history)에 자동 주입
   ↓
prompt → LLM 실행

핵심: memory는 "입력값"이 아니라 "시스템 레벨 상태"
"""


# =========================================================
# 8. RAG Chain (Retrieval + Generation)
# =========================================================

rag_chain = (
    RunnableParallel(
        # user question 그대로 전달
        question=lambda x: x["question"],

        # 핵심: FAISS retrieval 실행 (semantic search)
        context=


        # memory 전달 (RunnableWithMessageHistory가 자동 주입)
        chat_history=

    )
    | prompt  # prompt formatting
    | llm  # GPT inference
    | StrOutputParser()  # string output normalization
)


# =========================================================
# 9. Memory Store (세션 저장소)
# =========================================================

store =


def get_session_history(session_id: str):

    # session 없으면 생성 (lazy init pattern)
    if session_id not in store:
        store[session_id] = ChatMessageHistory()

    # session memory 반환
    return store[session_id]


# =========================================================
# 10. Memory Wrapper (LCEL + Session 연결)
# =========================================================

chain_with_memory = RunnableWithMessageHistory(
    rag_chain,  # base RAG chain

    get_session_history,  # session memory provider

    input_messages_key="question",  # user input key mapping

    history_messages_key="chat_history"  # prompt memory injection key
)


# =========================================================
# 11. 실행 함수 (사용자 인터페이스)
# =========================================================

def ask(question, session_id="user-1"):

    # 핵심: session 기반 memory + RAG 실행
    result =


    # 출력
    print("\n============================")
    print("QUESTION:", question)
    print("ANSWER:", result)
    print("============================")


# =========================================================
# 12. TEST (memory 동작 확인)
# =========================================================

ask("이 문서 핵심 요약해줘", "user-1")  # first interaction
ask("아까 내용을 더 쉽게 설명해줘", "user-1")  # memory-based follow-up

Saving ★ 서울특별시 스마트도시 및 정보화 기본계획(홈페이지 게시용).pdf to ★ 서울특별시 스마트도시 및 정보화 기본계획(홈페이지 게시용) (7).pdf

QUESTION: 이 문서 핵심 요약해줘
ANSWER: 서울특별시 스마트도시 및 정보화 기본계획(2021~2025)은 서울의 스마트도시 혁신을 위한 종합적인 전략을 담고 있습니다. 이 계획은 다음과 같은 주요 내용을 포함합니다:

1. **추진근거 및 배경**: 스마트도시 정책의 필요성과 배경 설명.
2. **국내외 동향**: 스마트도시 관련 국내외 최신 동향 분석.
3. **서울시 주요성과 및 정책 동향**: 서울시의 스마트도시 관련 성과와 정책 변화.
4. **현황 분석 및 추진방향**: 현재 상황 분석과 향후 추진 방향 제시.
5. **비전 및 추진전략**: 스마트도시의 비전과 이를 실현하기 위한 전략.
6. **전략별 세부과제**: 각 전략에 따른 세부 과제 목록과 주요 내용 요약.
   - **미래 스마트도시 혁신기반 조성**: 인프라 확충, 디지털 행정 혁신, 빅데이터 도시 조성.
   - **사람 중심 스마트도시 구현**: 비대면 서비스 확대, 포용적 도시 실현, 사이버 안전 강화.
   - **시민 체감 도시서비스 제공**: 스마트 모빌리티, 안전 도시 서비스, 디지털 경제 활성화.
7. **투자계획 및 성과지표**: 계획 실행을 위한 투자 계획과 성과 측정 지표.
8. **향후계획**: 향후 추진 계획 및 방향.

이 계획은 서울을 세계 최고의 스마트도시로 발전시키기 위한 구체적인 로드맵을 제시하고 있습니다.

QUESTION: 아까 내용을 더 쉽게 설명해줘
ANSWER: 서울특별시의 스마트도시 및 정보화 기본계획(2021~2025)은 서울을 더 스마트하고 편리한 도시로 만들기 위한 계획입니다. 이 계획의 주요 내용을 쉽게 설명하면 다음과 같습니다:

1. **왜 필요한가?**: 스마트도시 정책이 필요한 이유와 배경을 설명합니다.
2. **현재 상황**: 국내외 스마트도시의 최신 

대화를 저장할 함수 정의 - 수동

In [17]:

# =========================================================
# 1. Memory Store (세션별 대화 저장 공간)
# =========================================================

store = {}  # session_id → ChatMessageHistory 저장 딕셔너리

from langchain_community.chat_message_histories import ChatMessageHistory  # 대화 기록 객체
from langchain_core.runnables.history import RunnableWithMessageHistory  # LCEL memory wrapper
from langchain_core.runnables import RunnableParallel, RunnablePassthrough  # LCEL 병렬 실행/패스스루

from operator import itemgetter  # dict에서 값 추출용 (LCEL 보조)


# =========================================================
# 2. Session Manager (세션별 메모리 관리 함수)
# =========================================================

def get_session_history(session_id: str):

    print(f"[SESSION ID]: {session_id}")  # 현재 session 디버깅 출력

    # 해당 session이 없으면 새로운 memory 생성
    if session_id not in store:
        store[session_id] = ChatMessageHistory()

    # session memory 반환
    return store[session_id]


# =========================================================
# 3. Prompt 정의 (RAG 최종 입력 템플릿)
# =========================================================

from langchain_core.prompts import PromptTemplate  # prompt 템플릿 생성

prompt = PromptTemplate.from_template(
"""
You are a RAG assistant.  # 역할 정의

Chat History:  # 이전 대화 기록
{chat_history}

Question:  # 현재 질문
{question}

Context:  # 검색된 문서 내용
{context}

Answer:  # LLM이 생성할 답변
"""
)


# =========================================================
# 4. RAG Chain (Retrieval + Generation 핵심 구조)
# =========================================================

rag_chain = (
    RunnableParallel(
        # 질문 그대로 전달
        question=lambda x: x["question"],

        # retriever 실행 → context 생성
        context=lambda x: retriever.invoke(x["question"]),

        # memory에서 들어온 chat history 전달
        chat_history=lambda x: x.get("chat_history", [])
    )
    | prompt  # prompt에 입력값 매핑
    | llm  # LLM 실행
    | StrOutputParser()  # 문자열로 결과 변환
)


# =========================================================
# 5. Memory Wrapper (LCEL + Session Memory 연결)
# =========================================================

rag_with_history = RunnableWithMessageHistory(
    rag_chain,  # 실행할 RAG chain
    get_session_history,  # session_id → memory 반환 함수
    input_messages_key="question",  # 사용자 입력 key
    history_messages_key="chat_history"  # memory가 들어갈 key
)


# =========================================================
# 6. 실행 함수 (사용자 인터페이스 역할)
# =========================================================

def ask(question, session_id="user-1"):

    # RAG + Memory 실행
    result = rag_with_history.invoke(
        {"question": question},  # 입력값
        config={"configurable": {"session_id": session_id}}  # session 지정
    )

    # 출력 시작
    print("\n============================")
    print("QUESTION:", question)  # 질문 출력
    print("ANSWER:", result)  # 답변 출력
    print("============================")


# =========================================================
# 7. TEST (memory 동작 확인)
# =========================================================

ask("이 문서 핵심 내용 알려줘", "user-1")  # 1차 질문
ask("아까 내용을 더 쉽게 설명해줘", "user-1")  # memory 기반 follow-up 질문

[SESSION ID]: user-1

QUESTION: 이 문서 핵심 내용 알려줘
ANSWER: 서울특별시 스마트도시 및 정보화 기본계획(2021~2025)은 서울시의 스마트도시 발전을 위한 전략과 방향을 제시하는 문서입니다. 주요 내용은 다음과 같습니다:

1. **추진근거 및 배경**: 스마트도시 정책의 필요성과 배경 설명.
2. **국내외 동향**: 스마트도시 관련 국내외 최신 동향 분석.
3. **서울시 주요성과 및 정책 동향**: 서울시의 스마트도시 관련 성과와 정책 변화.
4. **현황 분석 및 추진방향**: 현재 상황 분석과 향후 추진 방향 제시.
5. **비전 및 추진전략**: 스마트도시의 비전과 이를 실현하기 위한 전략.
6. **전략별 세부과제**: 각 전략에 따른 세부 과제 목록과 주요 내용 요약.
   - **미래 스마트도시 혁신기반 조성**: 인프라 확충, 디지털 행정 혁신, 빅데이터 도시 조성.
   - **사람 중심 스마트도시 구현**: 비대면 서비스 확대, 포용 도시 실현, 사이버 안전.
   - **시민 체감 도시서비스 제공**: 스마트 모빌리티, 안전 도시 서비스, 디지털 경제 활성화.
7. **투자계획 및 성과지표**: 투자 계획과 성과를 측정할 지표 설정.
8. **향후 계획**: 향후 추진할 계획과 방향.

이 계획은 서울시를 세계 최고 수준의 스마트도시로 발전시키기 위한 종합적인 로드맵을 제공합니다.
[SESSION ID]: user-1

QUESTION: 아까 내용을 더 쉽게 설명해줘
ANSWER: 서울특별시의 스마트도시 및 정보화 기본계획(2021~2025)은 서울을 더 스마트하고 편리한 도시로 만들기 위한 계획입니다. 이 계획의 주요 내용을 쉽게 설명하면 다음과 같습니다:

1. **왜 스마트도시가 필요한가?**: 도시의 발전과 변화에 맞춰 스마트 기술을 도입해야 한다는 필요성을 설명합니다.

2. **국내외 동향**: 다른 나라와 서울의 스마트도시 관련 최신 동향을 살펴봅니다.

3. **서울의 성과와 정책 변화

첫 번째 질문 실행

In [13]:
# =========================================================
# 7. 핵심 수정된 invoke (네 코드 수정 버전)
# =========================================================

rag_with_history.invoke(
    {
        "question": "서울시 주요성과는?"
    },
    config={
        "configurable": {
            "session_id": "rag123"
        }
    }
)

[SESSION ID]: rag123


'서울시의 주요 성과는 다음과 같습니다:\n\n1. **전자정부 및 스마트시티 인프라**:\n   - 세계 주요 100대 도시 전자정부 평가에서 연속 8회 1위(2003~2019년).\n   - 스마트시티 평가에서 상위권 도시로 인정받음. 이든전략연구소의 평가에서 2021년 2위.\n   - Global Power City Index 2021에서 8위 수상.\n   - 바르셀로나 스마트시티 어워드 본상 수상(2019년).\n\n2. **미래 스마트도시 인프라 구축**:\n   - 공공장소 및 복지시설에 2만여 대의 공공 와이파이 구축.\n   - WiFi6 기반의 LTE보다 빠른 통신환경 조성.\n\n3. **빅데이터 시민서비스 확대**:\n   - 공공데이터 개방 확대: 2020년 6,617개에서 2021년 6,994개로 증가.\n   - 데이터 이용 건수도 120억 건에서 140억 건으로 증가.\n\n4. **시민소통 중심의 스마트서울맵 구축**:\n   - 코로나19 관련 도시생활지도 제작 및 공공 의료기관 정보 제공.\n\n이러한 성과들은 서울시가 스마트시티로서의 위상을 높이고, 시민의 삶의 질을 향상시키기 위한 다양한 노력을 반영하고 있습니다.'

이어진 질문 실행

In [14]:
rag_with_history.invoke(
    {
        "question": "스마트서울맵이 뭐야?"
    },
    config={
        "configurable": {
            "session_id": "rag123"
        }
    }
)

[SESSION ID]: rag123


'스마트서울맵은 서울시가 제공하는 디지털 플랫폼으로, 시민들이 도시의 다양한 정보를 쉽게 접근하고 활용할 수 있도록 돕는 서비스입니다. 이 맵은 코로나19와 같은 긴급 상황에서 도시 생활에 필요한 정보, 공공 의료기관의 위치, 그리고 다양한 시민 서비스에 대한 정보를 제공합니다. 또한, 시민 소통을 중심으로 구축되어 있어, 사용자들이 필요한 정보를 직관적으로 찾을 수 있도록 설계되었습니다. 스마트서울맵은 서울시의 스마트시티 정책의 일환으로, 시민의 삶의 질을 향상시키기 위한 다양한 노력을 반영하고 있습니다.'


기존 질문 기반 검색 RAG 시스템을 대화 상태를 기억하고 질문을 재해석하는 검색 Conversational RAG 시스템으로 개선.



- session memory 기반 stateful 구조
- chat history 기반 query rewriting
- context-aware retrieval
- multi-turn dialogue 지원


In [21]:
"""

User Question
    ↓
ChatMessageHistory (session_id 기반)
    ↓
query_rewriter_chain
    ↓
standalone_question 생성
    ↓
retriever (FAISS / BM25 / hybrid 가능)
    ↓
context 생성
    ↓
Prompt (chat_history + question + context)
    ↓
LLM
    ↓
Answer
    ↓
ChatMessageHistory 저장

"""

# =========================================================
# 1. Memory Store (세션별 대화 저장)
# =========================================================

from langchain_community.chat_message_histories import ChatMessageHistory  # 대화 저장 객체
from langchain_core.runnables.history import RunnableWithMessageHistory  # memory wrapper
from langchain_core.runnables import RunnableLambda  # LCEL 함수 래핑

store = {}  # session_id → ChatMessageHistory (in-memory DB)


def get_session_history(session_id):
    # 현재 어떤 사용자 세션인지 확인 (디버깅)
    print(f"[SESSION]: {session_id}")

    # 없으면 새 memory 생성 (lazy init)
    if session_id not in store:
        store[session_id] = ChatMessageHistory()

    return store[session_id]


# =========================================================
# 2. Query Rewriter (Conversational RAG 핵심)
# =========================================================

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

query_rewrite_prompt = ChatPromptTemplate.from_template(
"""
You are a query rewriting assistant.

Chat History:
{chat_history}

Question:
{question}

Rewrite into a standalone question.
"""
)

# 핵심: follow-up 질문을 "독립 질문"으로 변환
query_rewriter_chain =


# =========================================================
# 3. Context Builder (REFACTOR POINT)
# =========================================================

def build_context(inputs):

    # 1) 대화 기반 질문 정규화 (rewrite)
    standalone_question =



    # 2) Retriever 실행 (semantic search)
    docs =


    # 3) LLM 입력용 context 생성
    # (현재는 simple join → 확장 가능 지점)
    context =

    # 4) RAG input standardization
    return {


    }


# =========================================================
# 4. Prompt (Final RAG Prompt)
# =========================================================

rag_prompt = ChatPromptTemplate.from_template(
"""
You are a Conversational RAG assistant.

Chat History:
{chat_history}

Question:
{question}

Context:
{context}

Answer in Korean:
"""
)


# =========================================================
# 5. LLM (Generation Model)
# =========================================================

from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

llm_chain = ChatOpenAI(model="gpt-4o-mini", temperature=0)


# =========================================================
# 6. Conversational RAG Chain (Pipeline)
# =========================================================

conversational_rag_chain = (
       # rewrite + retrieval + formatting
    | rag_prompt                    # prompt injection
    | llm_chain                    # LLM inference
    | StrOutputParser()            # output normalize
)


# =========================================================
# 7. Memory Wrapper (Session Binding)
# =========================================================

conversational_rag = RunnableWithMessageHistory(
       # base RAG pipeline
       # session memory provider
    input_messages_key="question",
    history_messages_key="chat_history"
)


# =========================================================
# 8. Execution Function
# =========================================================

def ask(question, session_id="user-1"):

    # session 기반 RAG 실행
    result = conversational_rag.invoke(
        {"question": question},

        config=

    )

    print("\n==============================")
    print("QUESTION:", question)
    print("ANSWER:", result)
    print("==============================\n")


# =========================================================
# 9. TEST
# =========================================================

ask("이 문서 핵심 뭐야?", "user-1")
ask("그걸 더 쉽게 설명해줘", "user-1")
ask("예시 포함해서 다시 설명해줘", "user-1")

[SESSION]: user-1

QUESTION: 이 문서 핵심 뭐야?
ANSWER: 이 문서의 핵심은 저출산 문제 해결과 인구 감소 대응을 위한 정책으로, 기업 복지와 사회적 책임 강화를 포함하고 있다는 점입니다. 특히 부영그룹이 자녀 1명당 1억 원을 지원하는 출산 장려 정책을 발표한 것이 주요 내용입니다.

[SESSION]: user-1

QUESTION: 그걸 더 쉽게 설명해줘
ANSWER: 이 문서는 저출산 문제를 해결하고 인구가 줄어드는 것을 막기 위한 정책에 대해 이야기하고 있어. 기업들이 직원들을 더 잘 돌보고 사회에 책임을 다하자는 목표도 포함되어 있어. 특히 부영그룹은 아이를 낳는 가정에 대해 자녀 한 명당 1억 원을 지원하겠다고 발표했어. 쉽게 말하면, 아이를 더 많이 낳도록 도와주겠다는 거야.

[SESSION]: user-1

QUESTION: 예시 포함해서 다시 설명해줘
ANSWER: 이 문서는 저출산 문제를 해결하고 인구 감소에 대응하기 위한 정책에 대해 설명하고 있어. 예를 들어, 기업들이 직원들의 복지를 향상시키고 사회적 책임을 다하자는 목표도 포함되어 있어. 

부영그룹의 경우, 아이를 낳는 가정에 대해 자녀 한 명당 1억 원을 지원하겠다고 발표했어. 쉽게 말하면, 아이를 더 많이 낳도록 도와주겠다는 거야. 

예를 들어, 만약 한 부부가 두 명의 자녀를 낳으면, 부영그룹으로부터 총 2억 원을 지원받게 되는 거지. 이렇게 기업들이 출산을 장려하는 정책을 통해 저출산 문제를 해결하려고 하고 있어.



청크 분할 실험:
- 문서를 얼마나 잘게 나누느냐에 따라
→ 검색 정확도 vs 문맥 유지가 달라짐
- chunk_size / chunk_overlap 조정 → RAG 성능 비교

리트리버 비교:
- BM25(키워드 정확도)
- FAISS(임베딩 : 의미이해)
- 앙상블 → 두 다 합침(검색 정확도 강화)

RAG 체인:
- 검색 → 프롬프트 → LLM → 문자열 파싱 순서
- Query → Retriever → Context → Prompt → LLM → Answer

format_docs: 검색된 여러 청크를 LLM 입력용 하나의 문자열로 변환

test_rag 함수: 재사용 가능, 다양한 질문과 리트리버 조합 테스트 가능

In [17]:

# ================================================
# 1. 라이브러리 설치 (Colab / RAG 표준 스택)
# ================================================
# 역할:
# - LangChain RAG 전체 프레임워크
# - OpenAI LLM + Embedding
# - FAISS 벡터 DB (Dense search)
# - BM25 (Sparse search)
# - Web parsing + text processing

# 위에서 이미 설
!pip install -q \
    langchain \
    langchain-community \
    langchain-openai \
    langchain-experimental \
    openai \
    faiss-cpu \
    tiktoken \
    beautifulsoup4 \
    rank_bm25

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.6/133.6 kB 434.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.4/557.4 kB 1.8 MB/s eta 0:00:00


In [18]:
# langchain 버전 이슈로 고정 설치가 필요한 경우가 있음

!pip uninstall -y langchain langchain-community langchain-core
!pip install -q langchain==0.3.0 langchain-community langchain-openai

Found existing installation: langchain 1.3.11
Uninstalling langchain-1.3.11:
  Successfully uninstalled langchain-1.3.11
Found existing installation: langchain-community 0.4.2
Uninstalling langchain-community-0.4.2:
  Successfully uninstalled langchain-community-0.4.2
Found existing installation: langchain-core 1.4.8
Uninstalling langchain-core-1.4.8:
  Successfully uninstalled langchain-core-1.4.8
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.5/438.5 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.8/311.8 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5

In [1]:

# =========================================================
# 1. API KEY 설정
# =========================================================
# 역할: OpenAI API 인증 키를 환경 변수로 설정
# LangChain 전체 구성(OpenAI LLM / Embedding)에서 공통 사용

import os  # 환경 변수 설정을 위한 기본 라이브러리
from google.colab import userdata  # Colab 비밀 저장소 접근

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")  # API 키 환경 변수 등록


# =========================================================
# 2. 라이브러리 임포트
# =========================================================
# 역할: RAG 전체 파이프라인 구성 요소 로딩 (Loader / Splitter / Retriever / LLM)

import re  # 텍스트 정제용 정규표현식
import bs4  # HTML 파싱용 BeautifulSoup

from langchain_core.output_parsers import StrOutputParser  # LLM 출력 문자열 변환
from langchain_core.runnables import RunnablePassthrough  # 입력값 그대로 전달

from langchain_community.document_loaders import WebBaseLoader  # 웹 문서 로딩
from langchain_community.vectorstores import FAISS  # Dense vector DB (FAISS)

from langchain_openai import ChatOpenAI, OpenAIEmbeddings  # LLM + Embedding 모델

from langchain_community.retrievers import BM25Retriever  # Sparse keyword 검색
from langchain.retrievers.ensemble import EnsembleRetriever  # BM25 + FAISS Hybrid 검색

from langchain.text_splitter import RecursiveCharacterTextSplitter  # 텍스트 chunk 분할
from langchain_core.prompts import ChatPromptTemplate  # Prompt 템플릿


# =========================================================
# 3. 웹 문서 로딩
# =========================================================
# 역할: HTML 페이지 → LangChain Document 객체로 변환

loader = WebBaseLoader(
    web_paths=("https://n.news.naver.com/article/437/0000378416",),  # 뉴스 URL
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            "div",  # 특정 HTML div만 추출
            attrs={"class": ["newsct_article _article_body", "media_end_head_title"]},  # 본문 + 제목
        )
    ),
)

docs = loader.load()  # Document 리스트 생성


# =========================================================
# 4. 텍스트 전처리
# =========================================================
# 역할: HTML → 순수 텍스트 정리 (LLM 입력 품질 향상)

for doc in docs:  # 문서 단위 반복
    doc.page_content = re.sub(r"\s+", " ", doc.page_content).strip()  # 공백/줄바꿈 정리


# =========================================================
# 5. Chunking 함수 정의
# =========================================================
# 역할: chunk_size / overlap 변경 실험 가능하도록 구조화

def split_docs(docs, chunk_size, chunk_overlap):  # 문서 + 파라미터 입력
    splitter = RecursiveCharacterTextSplitter(  # 텍스트 분할기 생성
        chunk_size=chunk_size,  # 최대 chunk 길이
        chunk_overlap=chunk_overlap,  # 문맥 유지용 overlap
        separators=["\n\n", "\n", ".", "!", "?"]  # 문장 단위 분할 기준
    )
    return splitter.split_documents(docs)  # Document → chunk 리스트 반환


# =========================================================
# 6. Embedding 모델 정의
# =========================================================
# 역할: 텍스트를 벡터로 변환 (FAISS 검색용)

embedding = OpenAIEmbeddings(model="text-embedding-3-small")  # OpenAI embedding 모델


# =========================================================
# 7. RAG 구성 함수
# =========================================================
# 역할: chunk → vector DB → retriever 생성

def build_rag(split_docs):  # chunk 입력
    vectorstore = FAISS.from_documents(split_docs, embedding)  # FAISS 벡터 DB 생성

    bm25 = BM25Retriever.from_documents(split_docs)  # BM25 keyword retriever 생성
    bm25.k = 3  # 상위 3개 문서 검색

    faiss = vectorstore.as_retriever(search_kwargs={"k": 3})  # 의미 기반 retriever 생성

    ensemble = EnsembleRetriever(  # Hybrid retriever 생성
        retrievers=[bm25, faiss],  # BM25 + FAISS 결합
        weights=[0.5, 0.5]  # 동일 가중치 적용
    )

    return bm25, faiss, ensemble  # 3가지 retriever 반환


# =========================================================
# 8. Prompt + LLM 정의
# =========================================================
# 역할: 검색된 context 기반 답변 생성 규칙 정의

prompt = ChatPromptTemplate.from_template("""  # RAG 프롬프트 정의
너는 RAG 기반 QA 시스템이다.  # 역할 정의

Context만 사용해서 답변해라.  # 규칙
모르면 "문서에 없음"이라고 답해라.  # fallback 규칙

Context:
{context}

Question:
{question}

Answer:
""")

llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)  # 결정론적 답변 생성


# =========================================================
# 9. Document → 문자열 변환 함수
# =========================================================
# 역할: LLM 입력용 context 포맷 변환

def format_docs(docs):  # Document 리스트 입력
    return "\n\n".join(doc.page_content for doc in docs)  # 하나의 문자열로 변환


# =========================================================
# 10. RAG 실행 함수
# =========================================================
# 역할: retriever + question → LLM 결과 출력

def test_rag(question, retriever):  # 질문 + retriever 입력
    rag_chain = (  # LCEL 체인 구성 시작
        {
            "context": retriever | format_docs,  # 검색 → 문자열 변환
            "question": RunnablePassthrough()  # 질문 그대로 전달
        }
        | prompt  # prompt 적용
        | llm  # LLM 실행
        | StrOutputParser()  # 문자열 변환
    )

    result = rag_chain.invoke(question)  # 체인 실행

    print("=" * 60)  # 구분선 출력
    print("QUESTION:", question)  # 질문 출력
    print("\nANSWER:\n", result)  # 답변 출력
    print("=" * 60)  # 구분선 출력


# =========================================================
# 11. Chunk 실험 실행
# =========================================================
# 역할: chunk size별 RAG 성능 비교

split_300 = split_docs(docs, 300, 30)  # 작은 chunk
split_500 = split_docs(docs, 500, 50)  # 중간 chunk
split_800 = split_docs(docs, 800, 80)  # 큰 chunk

bm25_300, faiss_300, ens_300 = build_rag(split_300)  # 300 chunk retriever
bm25_500, faiss_500, ens_500 = build_rag(split_500)  # 500 chunk retriever
bm25_800, faiss_800, ens_800 = build_rag(split_800)  # 800 chunk retriever


# =========================================================
# 12. 실행 테스트
# =========================================================
# 역할: 동일 질문으로 chunk별 성능 비교

question = "부영그룹의 출산 장려 정책에 대해 설명해주세요"  # 테스트 질문

print("\n[CHUNK 300 - ENSEMBLE]")  # 출력 구분
test_rag(question, ens_300)  # 300 chunk 실행

print("\n[CHUNK 500 - ENSEMBLE]")  # 출력 구분
test_rag(question, ens_500)  # 500 chunk 실행

print("\n[CHUNK 800 - ENSEMBLE]")  # 출력 구분
test_rag(question, ens_800)  # 800 chunk 실행


[CHUNK 300 - ENSEMBLE]
QUESTION: 부영그룹의 출산 장려 정책에 대해 설명해주세요

ANSWER:
 부영그룹의 출산 장려 정책은 2021년 이후 태어난 직원 자녀에게 1억원을 지원하고 앞으로도 이 정책을 이어가기로 했습니다.

[CHUNK 500 - ENSEMBLE]
QUESTION: 부영그룹의 출산 장려 정책에 대해 설명해주세요

ANSWER:
 부영그룹은 2021년 이후 태어난 직원 자녀에게 1억원을 지원하고, 이 정책을 앞으로도 이어갈 계획이다. 연년생과 쌍둥이 자녀가 있는 경우에는 총 2억원을 받을 수 있으며, 셋째까지 낳는 경우에는 국민주택을 제공할 예정이다. 해당 출산장려금은 직원들의 세금 부담을 고려해 정부가 면세해달라는 제안도 나왔다.

[CHUNK 800 - ENSEMBLE]
QUESTION: 부영그룹의 출산 장려 정책에 대해 설명해주세요

ANSWER:
 부영그룹은 2021년 이후 태어난 직원 자녀에게 1억원을 지원하고, 연년생과 쌍둥이 자녀가 있는 경우에는 총 2억원을 받을 수 있습니다. 또한, 셋째까지 아이를 낳는 경우에는 국민주택을 제공할 계획이 있습니다.
